In [ ]:
!pip install transformers datasets accelerate evaluate

In [ ]:
from datasets import load_dataset

# Load IMDB dataset
dataset = load_dataset("Vinod-IE/IDMBreview")

# View the IMDB dataset
print(dataset)
print(dataset['train'][0])  # See what a sample looks like

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load a small, fast model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Tokenize the text
def tokenize_function(examples):
    return tokenizer(examples['review'], padding='max_length', truncation=True, max_length=512)

# Apply to dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Map string sentiment labels to integers (0 for negative, 1 for positive)
def map_labels(examples):
    return {'labels': 0 if examples['sentiment'] == 'negative' else 1}

tokenized_datasets = tokenized_datasets.map(map_labels)

# Remove original 'sentiment' column and 'review_length' if not needed
tokenized_datasets = tokenized_datasets.remove_columns(['sentiment', 'review_length'])

# Create smaller subsets for faster training
small_train = tokenized_datasets['train'].shuffle(seed=42).select(range(1000))
small_test = tokenized_datasets['validation'].shuffle(seed=42).select(range(200))

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate # Import the evaluate library

# Define how to compute metrics
metric = evaluate.load("accuracy") # Use evaluate.load()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Set training parameters
training_args = TrainingArguments(
    output_dir="./results",
    # Removed evaluation_strategy as it caused a TypeError
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=10, # Increased from 3 to 10
    weight_decay=0.01,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

# Train!
trainer.train()

model.save_pretrained("./fine_tuned_sentiment_model")
tokenizer.save_pretrained("./fine_tuned_sentiment_model")

In [ ]:
# Test the fine-tuned model
results = trainer.evaluate()
print(f"Fine-tuned model accuracy: {results['eval_accuracy']:.2%}")

# Test on a few examples
test_texts = [
    "This movie was amazing! Best film I've seen all year.",
    "Terrible waste of time. I want my money back.",
]

inputs = tokenizer(test_texts, padding=True, truncation=True, return_tensors="pt")
# Move input tensors to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=-1)

for text, pred in zip(test_texts, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"Text: {text}")
    print(f"Prediction: {sentiment}\n")